<a href="https://colab.research.google.com/github/PETUFCEstatistica/Oficina-Github-Colab/blob/main/codigos/01-limpeza_de_dados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "AB_NYC_2019.csv"

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "dgomonov/new-york-city-airbnb-open-data",
    file_path
)

df.to_csv("../dados/brutos/dados-brutos.csv", index=False)

/var/folders/3m/wf190yyn0cxfp4lf7rvctf380000gn/T/ipykernel_42532/1148085239.py:7: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


In [10]:
#INFORMAÇÕES SOBRE A BASE DE DADOS
df.info()
print(df.isna().sum())
print(df.duplicated().sum())
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48895 entries, 0 to 48894
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              48895 non-null  int64  
 1   name                            48879 non-null  object 
 2   host_id                         48895 non-null  int64  
 3   host_name                       48874 non-null  object 
 4   neighbourhood_group             48895 non-null  object 
 5   neighbourhood                   48895 non-null  object 
 6   latitude                        48895 non-null  float64
 7   longitude                       48895 non-null  float64
 8   room_type                       48895 non-null  object 
 9   price                           48895 non-null  int64  
 10  minimum_nights                  48895 non-null  int64  
 11  number_of_reviews               48895 non-null  int64  
 12  last_review                     

,id,host_id,latitude,longitude,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365
count,4.889500e+04,4.889500e+04,48895.000000,48895.000000,48895.000000,48895.000000,48895.000000,38843.000000,48895.000000,48895.000000
mean,1.901714e+07,6.762001e+07,40.728949,-73.952170,152.720687,7.029962,23.274466,1.373221,7.143982,112.781327
std,1.098311e+07,7.861097e+07,0.054530,0.046157,240.154170,20.510550,44.550582,1.680442,32.952519,131.622289
min,2.539000e+03,2.438000e+03,40.499790,-74.244420,0.000000,1.000000,0.000000,0.010000,1.000000,0.000000
25%,9.471945e+06,7.822033e+06,40.690100,-73.983070,69.000000,1.000000,1.000000,0.190000,1.000000,0.000000
50%,1.967728e+07,3.079382e+07,40.723070,-73.955680,106.000000,3.000000,5.000000,0.720000,1.000000,45.000000
75%,2.915218e+07,1.074344e+08,40.763115,-73.936275,175.000000,5.000000,24.000000,2.020000,2.000000,227.000000
max,3.648724e+07,2.743213e+08,40.913060,-73.712990,10000.000000,1250.000000,629.000000,58.500000,327.000000,365.000000


In [11]:
#CONVERTER TIPOS
df['last_review'] = pd.to_datetime(df['last_review'], errors='coerce')
df['neighbourhood_group'] = df['neighbourhood_group'].astype('category')
df['room_type'] = df['room_type'].astype('category')

In [12]:
df['name'] = df['name'].fillna('sem nome')
df["reviews_per_month"] = df["reviews_per_month"].fillna(0)
df['host_name'] = df['host_name'].fillna('desconhecido')
df = df.drop_duplicates()

In [13]:
#ANÁLISE DA DISTRIBUIÇÂO DOS DADOS
df["price"].describe()
df["price"].quantile([0.5, 0.9, 0.95, 0.99])

0.50    106.0
0.90    269.0
0.95    355.0
0.99    799.0
Name: price, dtype: float64

In [14]:
#TRATAMENTO DE OUTLIERS
q1, q3 = df['price'].quantile([0.25, 0.75])
limite = q3 + 1.5 * (q3 - q1)
df_limpo = df[(df['price'] <= limite) & (df['price'] > 0)].copy()

df_limpo.to_csv("../dados/limpos/dados-limpos.csv", index=False)

In [15]:
#REMOVIDOS
removidos = len(df) - len(df_limpo)
print(f"Limite: {limite:.0f}")
print(f"Antes: {len(df)} | Depois: {len(df_limpo)} | Removidos: {removidos} ({removidos / len(df):.1%})")

Limite: 334
Antes: 48895 | Depois: 45912 | Removidos: 2983 (6.1%)
